# You have sixty dollars. Where do they go?

A fitted surface is not a decision. The decision is an allocation — how much of the budget to
each treatment — and the two ways it usually gets made are proportional-to-last-year and
all-into-whatever-scored-highest. Both ignore the shape of the curve, which is the entire
content of the model that was just fitted.

Classical RSM moves on a fitted surface: the gradient and Hessian of `forward()` (jax when
available, unit-invariant finite differences otherwise), steepest ascent, canonical analysis of
the stationary point, and the budget-constrained allocator with its effort frontier.

In [ ]:
import numpy as np

from axiom.core import D, Outcome, Treatment, is_failure
from axiom.surface import (
    Allocation, AllocationMethod, AscentPath, Bounds, Frontier, HillKernel, Method, Objective,
    StationaryPoint, Surface, SurfaceSpec, allocate, canonical_analysis, frontier, gradient, hessian,
    steepest_ascent,
)

from axiom.display import enable, table

import sys; sys.path[:0] = ["..", "../.."]  # nbs/ is on the path either way
from _style import BLUE, ORANGE, annotate, caption, compare, curve_band, heat, mark_x, points

enable();  # every axiom result renders itself from here on

In [ ]:
spec = SurfaceSpec(
    name="two_hill",
    treatments=(Treatment(name="a", dimension=D.currency, unit="USD"), Treatment(name="b", dimension=D.currency, unit="USD")),
    outcome=Outcome(name="y", dimension=D.outcome),
    kernels={"a": HillKernel(reference_dose=50.0), "b": HillKernel(reference_dose=20.0)},
)
surface = Surface(spec)
theta = {"alpha": 1.0, "k_a": 50.0, "s_a": 2.0, "beta_a": 10.0, "k_b": 20.0, "s_b": 1.5, "beta_b": 5.0, "sigma": 1.0}
bounds = Bounds(treatments=("a", "b"), low=(0.0, 0.0), high=(200.0, 80.0))
method: Method = "auto"
print(gradient(surface, theta, {"a": 30.0, "b": 10.0}, method=method))
print(np.round(hessian(surface, theta, {"a": 30.0, "b": 10.0}, method=method), 5))

In [ ]:
from axiom.display import show

path = steepest_ascent(surface, theta, {"a": 10.0, "b": 5.0}, step=5.0, n_steps=20, bounds=bounds)
if isinstance(path, AscentPath):
    print(path.n, path.stop, path.best(), round(path.values[-1], 3))
else:
    show(path)

In [ ]:
a_grid = np.linspace(0.0, 200.0, 21)
b_grid = np.linspace(0.0, 80.0, 17)
mesh = [[float(np.ravel(surface.forward({"a": np.array([av]), "b": np.array([bv])}, theta))[0])
         for av in a_grid] for bv in b_grid]

fig = heat(
    mesh, [f"{v:.0f}" for v in a_grid], [f"{v:.0f}" for v in b_grid],
    text_fmt="",
    colorbar_title="outcome",
    title="The surface the ascent is walking on",
    subtitle="expected outcome over the two-treatment dose space at the fitted parameters",
    x_title="dose of a (USD)", y_title="dose of b (USD)",
    height=460,
)
caption(fig, "No interior peak: the surface rises towards the top-right corner and flattens. "
             "That is what saturation looks like in two dimensions, and it is why the next "
             "section's canonical analysis reports a ridge rather than an optimum.")

In [ ]:
if isinstance(path, AscentPath):
    fig = points(
        {"ascent path": ([p[0] for p in path.points], [p[1] for p in path.points])},
        title="Twenty steps up the gradient",
        subtitle="from (10, 5), step 5.0, clipped to the box — each point is one gradient evaluation",
        x_title="dose of a (USD)", y_title="dose of b (USD)",
        height=420,
    )
    annotate(fig, path.points[0][0], path.points[0][1], "start")
    annotate(fig, path.points[-1][0], path.points[-1][1], f"stop: {path.stop}")
    caption(fig, "The path bends as the cheaper treatment saturates and the gradient turns "
                 "towards the other one. A proportional split would have walked in a straight "
                 "line and stopped somewhere lower.")
    fig

A saturating surface has no interior maximum, so canonical analysis reports a ridge or a
`Unsupported`; on a quadratic it finds and classifies the stationary point.

In [ ]:
sp = canonical_analysis(surface, theta, {"a": 30.0, "b": 10.0}, method=method)
print(sp if is_failure(sp) else (sp.kind, sp.point, sp.eigenvalues))

## Allocation under a budget

`allocate` maximizes the expected outcome subject to a total-dose budget and box bounds
(SLSQP; the cvxpy path is optional). Non-convergence is a typed `Unsupported`, never a number —
the parent repo once shipped an allocator that swallowed SLSQP's failure flag.

In [ ]:
obj: Objective = "mean"
meth: AllocationMethod = "slsqp"
alloc = allocate(surface, theta, budget=60.0, bounds=bounds, objective=obj, method=meth, seed=0)
if isinstance(alloc, Allocation):
    print(alloc.doses, round(alloc.expected_outcome, 3), alloc.status, round(alloc.total_dose, 3))
else:
    show(alloc)
print(allocate(surface, theta, budget=60.0, bounds=bounds, maxiter=1))

In [ ]:
if isinstance(alloc, Allocation):
    budget = 60.0
    def outcome_of(doses):
        return float(np.ravel(surface.forward({k: np.array([v]) for k, v in doses.items()}, theta))[0])

    rules = {
        "everything into a": {"a": budget, "b": 0.0},
        "split it evenly": {"a": budget / 2, "b": budget / 2},
        "proportional to last year (70/30)": {"a": 0.7 * budget, "b": 0.3 * budget},
        "allocate()": dict(alloc.doses),
    }
    fig = compare(
        [f"{label}  →  a={d['a']:.0f}, b={d['b']:.0f}" for label, d in rules.items()],
        [outcome_of(d) for d in rules.values()],
        highlight=f"allocate()  →  a={alloc.doses['a']:.0f}, b={alloc.doses['b']:.0f}",
        value_fmt="{:.2f}",
        title="Four ways to spend the same sixty dollars",
        subtitle="expected outcome at the fitted parameters, total dose held at 60 in every row",
        x_title="expected outcome",
    )
    caption(fig, "Putting the whole budget into the better-scoring treatment is the worst of "
                 "the four, and an even split is barely better: both ignore that treatment b "
                 "saturates at a fifth of the dose a does. The margin over a sensible "
                 "70/30 guess is small — which is the honest reading. The allocator earns its "
                 "keep at larger budgets, and on more than two treatments.")
    fig

In [ ]:
fr = frontier(surface, theta, budgets=[10.0, 30.0, 60.0, 120.0], bounds=bounds, seed=0)
if isinstance(fr, Frontier):
    print(fr.as_frame())
    print("shadow prices:", np.round(fr.shadow_prices(), 4))
else:
    show(fr)

In [ ]:
wide = frontier(surface, theta, budgets=[5.0, 15.0, 30.0, 60.0, 100.0, 150.0, 220.0, 280.0], bounds=bounds, seed=0)
if isinstance(wide, Frontier):
    frame = wide.as_frame()
    fig = curve_band(
        frame["budget"], frame["expected_outcome"],
        label="best achievable",
        title="What the next dollar buys",
        subtitle="the effort frontier: expected outcome under an optimal split at each budget",
        x_title="total budget (USD)", y_title="expected outcome",
    )
    prices = np.round(wide.shadow_prices(), 4)
    knee = int(np.argmax(prices < prices[0] / 4)) if np.any(prices < prices[0] / 4) else len(prices) - 1
    mark_x(fig, float(frame["budget"].iloc[knee]), text="shadow price down 4×")
    caption(fig, f"Shadow prices from {prices[0]:.3f} down to {prices[-1]:.3f} outcome per "
                 f"dollar across this range. The curve is the argument for a budget number, "
                 f"and it is the same object nbs/design/06 turns into a portfolio decision.")
    fig

## What this bought you

A budget split that reads the fitted curve instead of last year's split, a frontier that says
what the next dollar is worth rather than only where the optimum is, and — the part the parent
repo got wrong — a typed refusal when the optimizer does not converge, instead of whatever
point it happened to stop at.